# M0.3 · Türev, gradyan, Jacobian, zincir kuralı

Backprop'un tamamı bu modülde: küçük bir hesap grafiğinde ileri ve geri geçişi elle yapıyor, sayısal türevle doğruluyor, NumPy'da tüm kısmi türevleri tek çağrıda alıyor, autograd'a bırakıyor ve en sonda yüzlerce noktanın Jacobian'ını `vmap` ile birden hesaplıyorsun.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m0-3-chain-rule.html](https://m-gonul.github.io/pytorch-lab/m0-3-chain-rule.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m0_3_chain_rule.py
CHECKS = {
 "b": "import math\nimport random\nfrom labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"numpy/torch kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def fb():\n        out = need(ns, \"f_backward\")(2.0, 3.0)\n        ensure(isinstance(out, (tuple, list)) and len(out) == 3, \"(out, dfdx, dfdy) üçlüsü döndürmeli\")\n        close(out, [15, 3, 8], what=\"(f, ∂f/∂x, ∂f/∂y)\")\n    R.test(\"elle örnek: (15, 3, 8)\", fb)\n\n    def fb_rand():\n        rng = random.Random(2)\n        for _ in range(10):\n            x, y = rng.uniform(-4, 4), rng.uniform(-4, 4)\n            close(need(ns, \"f_backward\")(x, y), [(x + y) * y, y, x + 2 * y], what=f\"(x, y) = ({x:.2f}, {y:.2f})\")\n    R.test(\"rastgele noktalar: ∂f/∂x = y, ∂f/∂y = x + 2y\", fb_rand)\n\n    def ng():\n        g = need(ns, \"num_grad\")\n        close(g(lambda x, y: (x + y) * y, [2.0, 3.0]), [3, 8], rtol=1e-6, atol=1e-6)\n        fn = lambda a, b, c: math.sin(a) * b ** 2 + math.exp(c) * a\n        xs = [0.7, -1.3, 0.2]\n        want = [math.cos(0.7) * 1.69 + math.exp(0.2), 2 * math.sin(0.7) * -1.3, math.exp(0.2) * 0.7]\n        close(g(fn, xs), want, rtol=1e-6, atol=1e-6, what=\"sin(a)·b² + eᶜ·a\")\n        ensure(xs == [0.7, -1.3, 0.2], \"num_grad girdi listesini değiştirdi\")\n    R.test(\"num_grad: polinom ve sin/exp fonksiyonu, girdi değişmiyor\", ng)\n",
 "c": "import numpy as np\nfrom labcheck import close, no_loops, no_imports, need, ensure\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü/comprehension yok\", lambda: no_loops(src))\n    R.test(\"torch kullanılmıyor\", lambda: no_imports(src, {\"torch\"}))\n    f = lambda X: (X[:, 0] + X[:, 1]) * X[:, 1]\n    g = lambda X: np.stack([X[:, 0] * X[:, 1], X[:, 0] + X[:, 1] ** 2], axis=1)\n    calls = []\n\n    def counting(F):\n        def w(X):\n            calls.append(np.asarray(X).shape)\n            return F(X)\n        return w\n\n    def grad2():\n        out = need(ns, \"num_grad_np\")(counting(f), np.array([2.0, 3.0]))\n        close(out, [3, 8], rtol=1e-6, atol=1e-6)\n        ensure(len(calls) == 1, f\"f {len(calls)} kez çağrıldı; tek batch çağrısı bekleniyordu\")\n        return \"f tek çağrıda (4×2 batch) değerlendirildi\"\n    R.test(\"num_grad_np: [3, 8], tek çağrı\", grad2)\n\n    def grad_n():\n        A = np.random.default_rng(0).normal(size=(6, 6)); A = A + A.T\n        q = lambda X: 0.5 * np.einsum(\"ki,ij,kj->k\", X, A, X) + np.sin(X).sum(1)\n        x = np.linspace(-1, 1, 6)\n        close(need(ns, \"num_grad_np\")(q, x), A @ x + np.cos(x), rtol=1e-6, atol=1e-6, what=\"½xᵀAx + Σ sin xᵢ\")\n    R.test(\"6 boyutlu kuadratik + sin\", grad_n)\n\n    def jac():\n        J = need(ns, \"num_jacobian_np\")(g, np.array([2.0, 3.0]))\n        ensure(np.asarray(J).shape == (2, 2), f\"shape {np.asarray(J).shape}, (m, n) = (2, 2) bekleniyordu\")\n        close(J, [[3, 2], [1, 6]], rtol=1e-6, atol=1e-6)\n        W = np.random.default_rng(1).normal(size=(3, 4))\n        J2 = need(ns, \"num_jacobian_np\")(lambda X: X @ W.T, np.ones(4))\n        ensure(np.asarray(J2).shape == (3, 4), f\"ℝ⁴→ℝ³ için shape {np.asarray(J2).shape}, (3, 4) bekleniyordu\")\n        close(J2, W, rtol=1e-6, atol=1e-6, what=\"lineer g(x) = Wx'in Jacobian'ı W\")\n    R.test(\"num_jacobian_np: [[3,2],[1,6]] ve lineer haritada J = W\", jac)\n",
 "d": "import torch\nfrom labcheck import close, need, ensure, uses_any\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n\n    def ga():\n        out = need(ns, \"grads_autograd\")(2.0, 3.0)\n        ensure(isinstance(out, (tuple, list)) and len(out) == 2, \"(df/dx, df/dy) çifti döndürmeli\")\n        close(out, [3, 8])\n        close(need(ns, \"grads_autograd\")(-1.5, 0.5), [0.5, -0.5])\n    R.test(\"grads_autograd: (3, 8) ve (−1.5, 0.5) noktası\", ga)\n    R.test(\"backward kullanılıyor\", lambda: ensure(uses_any(src, [\"backward\", \"grad\"]), \"türevi formülle yazma; autograd'a bırak\"))\n\n    def jt():\n        J = need(ns, \"jacobian_torch\")(torch.tensor([2.0, 3.0]))\n        ensure(torch.is_tensor(J) and tuple(J.shape) == (2, 2), \"(2, 2) tensör bekleniyordu\")\n        close(J, [[3, 2], [1, 6]])\n        close(need(ns, \"jacobian_torch\")(torch.tensor([-1.0, 0.5])), [[0.5, -1], [1, 1]], what=\"(−1, 0.5) noktası\")\n    R.test(\"jacobian_torch: [[3,2],[1,6]]\", jt)\n",
 "e": "import torch\nfrom labcheck import close, no_loops, need, ensure, bench, fmt_time\n\n\ndef check(ns, R):\n    src = ns[\"__source__\"]\n    R.test(\"döngü yok\", lambda: no_loops(src))\n    g = torch.Generator().manual_seed(0)\n    P = torch.randn(40, 2, generator=g, dtype=torch.float64)\n    x, y = P[:, 0], P[:, 1]\n\n    def bg():\n        G = need(ns, \"batch_grad\")(P)\n        ensure(tuple(G.shape) == (40, 2), f\"shape {tuple(G.shape)}, (40, 2) bekleniyordu\")\n        close(G, torch.stack([y, x + 2 * y], 1), what=\"∇f = (y, x + 2y)\")\n    R.test(\"batch_grad: 40 noktada (y, x + 2y)\", bg)\n\n    def bj():\n        J = need(ns, \"batch_jacobian\")(P)\n        ensure(tuple(J.shape) == (40, 2, 2), f\"shape {tuple(J.shape)}, (40, 2, 2) bekleniyordu\")\n        want = torch.stack([torch.stack([y, x], 1), torch.stack([torch.ones_like(x), 2 * y], 1)], 1)\n        close(J, want, what=\"J = [[y, x], [1, 2y]]\")\n    R.test(\"batch_jacobian: (40, 2, 2)\", bj)\n    if \"batch_jacobian\" not in ns:\n        return\n    from torch.autograd.functional import jacobian\n    gv = lambda p: torch.stack([p[0] * p[1], p[0] + p[1] ** 2])\n    Pb = torch.randn(512, 2)\n    t_loop = bench(lambda: torch.stack([jacobian(gv, p) for p in Pb]), repeat=3, number=1)\n    t_vm = bench(lambda: ns[\"batch_jacobian\"](Pb), repeat=3)\n    R.table(\"512 noktanın Jacobian'ı\", [\"yöntem\", \"süre\", \"döngüye göre\"],\n            [[\"for + autograd.functional.jacobian\", fmt_time(t_loop), \"1×\"],\n             [\"senin batch_jacobian\", fmt_time(t_vm), f\"{t_loop / t_vm:,.0f}× hızlı\"]],\n            note=\"Döngüde her nokta için ayrı bir grafik kurulur ve m = 2 ayrı backward yapılır; vmap tüm bunları batch boyutlu tek işlemlere çevirir.\")\n"
}

In [ ]:
import importlib, labcheck, checks_m0_3_chain_rule as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

Bir sinir ağı, basit işlemlerden oluşan uzun bir bileşik fonksiyondur. Zincir kuralı, bu bileşiğin türevini parçaların yerel türevlerinden kurmayı sağlar. İfadeyi bir hesap grafiği olarak çizersek:

$$
x,\;y \;\longrightarrow\; a = x+y \;\longrightarrow\; f = a\cdot y
$$

**İleri geçiş** düğüm değerlerini soldan sağa hesaplar. **Geri geçiş** sağdan sola giderken her düğümde "yukarıdan gelen gradyan × yerel türev" çarpımını yapar. Bir değişken birden fazla düğüme giriyorsa, o değişkene gelen katkılar **toplanır**. Bütün backpropagation bu iki kuraldan ibaret.

Vektör değerli $g:\mathbb{R}^n\to\mathbb{R}^m$ için türevlerin tamamı bir matristir, Jacobian: $J_{ij}=\partial g_i/\partial x_j$, shape'i $(m\times n)$.

## 1 · Elle

Grafiği kâğıda çiz: iki girdi düğümü ($x$, $y$), bir toplama düğümü ($a$), bir çarpma düğümü ($f$). $y$'den **iki** ok çıktığına dikkat et.

Geri geçişte $\partial f/\partial f = 1$ ile başla ve her düğümde aşağıdaki tabloyu kullan:

| düğüm | yerel türevler |
|---|---|
| $a = x + y$ | $\partial a/\partial x = 1$, $\partial a/\partial y = 1$ |
| $f = a\cdot y$ | $\partial f/\partial a = y$, $\partial f/\partial y = a$ |

Sonra aynı sayıları sayısal türevle (ileri ve geri birer adım) doğrula. Son olarak iki çıktılı bir fonksiyonun $2\times2$ Jacobian'ını doldur.


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**İleri geçiş: $f(x,y)=(x+y)\cdot y$, $x=2$, $y=3$; ara düğüm $a=x+y$**

- `a`: $a = x+y$
- `f`: $f = a\cdot y$

**Geri geçiş (yerel türev × yukarıdan gelen)**

- `dfda`: $\partial f/\partial a$
- `dfdy_dir`: $\partial f/\partial y$ yalnız çarpım düğümü üzerinden
- `dfdx`: $\partial f/\partial x$
- `dfdy`: $\partial f/\partial y$ (toplam)

**Sayısal kontrol, $h=0.1$: $\frac{f(x+h)-f(x-h)}{2h}$**

- `num_dx`: $x$'e göre
- `num_dy`: $y$'ye göre

**Jacobian: $g(x,y)=[\,xy,\;x+y^2\,]$, $(x,y)=(2,3)$; $J_{ij}=\partial g_i/\partial x_j$**

- `j11`: $\partial g_1/\partial x$
- `j12`: $\partial g_1/\partial y$
- `j21`: $\partial g_2/\partial x$
- `j22`: $\partial g_2/\partial y$

In [ ]:
cevaplar = dict(
    a=None,
    f=None,
    dfda=None,
    dfdy_dir=None,
    dfdx=None,
    dfdy=None,
    num_dx=None,
    num_dy=None,
    j11=None,
    j12=None,
    j21=None,
    j22=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"a":[5,0],"f":[15,0],"dfda":[3,0],"dfdy_dir":[5,0],"dfdx":[3,0],"dfdy":[8,0],"num_dx":[3,0.001],"num_dy":[8,0.001],"j11":[3,0],"j12":[2,0],"j21":[1,0],"j22":[6,0]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

İki şey yaz:

- `f_backward(x, y)` — ileri geçişi ara değişkenlerle (`a = x + y`, `out = a * y`) yap, sonra geri geçişi **elle**, düğüm düğüm yaz. `(out, dfdx, dfdy)` döndür. Formülü doğrudan yazma ($\partial f/\partial y = x+2y$); grafiği takip et — M3.1'de bunu otomatikleştireceksin.
- `num_grad(f, xs, h=1e-5)` — herhangi bir `f(*xs)` için merkezi farkla kısmi türev listesi. `xs`'i değiştirme; her koordinat için bir kopya oluştur (`if`/`for` serbest).

`numpy` ve `torch` yasak.

In [ ]:
%%writefile stage_b.py
def f_backward(x, y):
    """f = (x + y) * y. (out, df/dx, df/dy) döndür — geri geçişi düğüm düğüm yaz."""
    # ileri
    a = x + y
    out = a * y
    # geri: dout = 1 ile başla
    dout = 1.0
    # da = ?        (çarpım düğümünden a'ya)
    # dy = ?        (çarpım düğümünden y'ye, doğrudan)
    # dx = ?        (toplama düğümünden x'e)
    # dy += ?       (toplama düğümünden y'ye — ikinci yol!)
    raise NotImplementedError


def num_grad(f, xs, h=1e-5):
    """f(*xs)'in her koordinata göre merkezi fark türevi (liste)."""
    ...


print(f_backward(2.0, 3.0))                       # (15.0, 3.0, 8.0)
print(num_grad(lambda x, y: (x + y) * y, [2.0, 3.0]))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

Sayısal gradyanın $n$ koordinat için $2n$ değerlendirme istediğini gördün. Bunları döngü yerine **tek bir batch** olarak yapabiliriz: $x$'in $2n$ kopyasını bir matrise dizip fonksiyonu bir kez çağırırız.

$$
X=\begin{bmatrix} x + hI \\ x - hI\end{bmatrix}\in\mathbb{R}^{2n\times n},\qquad
\nabla f \approx \frac{f(X_{1:n}) - f(X_{n+1:2n})}{2h}
$$

Bu aşamadaki fonksiyonlar satır satır çalışır: `f(X)` `(k, n) → (k,)`, `g(X)` `(k, n) → (k, m)`.

- `num_grad_np(f, x, h=1e-5)` → `(n,)`
- `num_jacobian_np(g, x, h=1e-5)` → `(m, n)`; aynı hile, sonuç transpoze edilmeli.

**Döngü yok.** `np.eye`, `np.vstack`/`np.concatenate` yeterli.

In [ ]:
%%writefile stage_c.py
import numpy as np


def num_grad_np(f, x, h=1e-5):
    """f: (k, n) → (k,). x: (n,). Döngüsüz merkezi fark gradyanı → (n,)."""
    ...


def num_jacobian_np(g, x, h=1e-5):
    """g: (k, n) → (k, m). x: (n,). → (m, n) Jacobian."""
    ...


f = lambda X: (X[:, 0] + X[:, 1]) * X[:, 1]
g = lambda X: np.stack([X[:, 0] * X[:, 1], X[:, 0] + X[:, 1] ** 2], axis=1)
print(num_grad_np(f, np.array([2.0, 3.0])))        # [3. 8.]
print(num_jacobian_np(g, np.array([2.0, 3.0])))    # [[3. 2.] [1. 6.]]

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

Şimdi grafiği autograd kursun. PyTorch her işlemi kaydeder, `.backward()` grafiği ters topolojik sıralama ile gezer ve `x.grad`'a yazar; bir değişken iki yoldan kullanıldıysa katkıları kendisi toplar.

- `grads_autograd(x, y)` — iki skaleri `requires_grad=True` tensöre çevir, $f$'yi hesapla, `backward()` çağır, `(dfdx, dfdy)` döndür.
- `jacobian_torch(p)` — `p = tensor([x, y])` için $g(p)=[\,xy,\;x+y^2\,]$'nin Jacobian'ı. `torch.autograd.functional.jacobian` ya da `torch.func.jacrev` kullan.

In [ ]:
%%writefile stage_d.py
import torch


def grads_autograd(x, y):
    """f = (x + y) * y için (df/dx, df/dy) — backward ile."""
    ...


def jacobian_torch(p):
    """p = tensor([x, y]) → g(p) = [x*y, x + y**2]'nin (2, 2) Jacobian'ı."""
    ...


print(grads_autograd(2.0, 3.0))                        # (tensor(3.), tensor(8.))
print(jacobian_torch(torch.tensor([2.0, 3.0])))

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

Aynı türevi $N$ farklı noktada istiyoruz (ör. bir grafikte gradyan alanı çizmek, ya da örnek başına türev). Döngüyle $N$ kez backward çağırmak yerine iki yol:

1. **Toplam hilesi:** noktalar birbirinden bağımsızsa $\sum_k f(p_k)$'nin $p_k$'ya göre gradyanı tam $\nabla f(p_k)$'dır. Tek `backward`.
2. **`torch.func`:** tek nokta için yazılmış dönüşümü vmap ile batch'e genişlet:

```python
from torch.func import grad, jacrev, vmap
vmap(grad(f_scalar))(P)    # (N, 2)
vmap(jacrev(g_vec))(P)     # (N, 2, 2)
```

- `batch_grad(P)` — `P: (N, 2)` → her satırda $(\partial f/\partial x,\partial f/\partial y)$.
- `batch_jacobian(P)` — `(N, 2) → (N, 2, 2)`.

İkisinde de **döngü yok**. Kontrol, 512 nokta için `for` + `jacobian` ile `vmap(jacrev)`'i yarıştırır.

In [ ]:
%%writefile stage_e.py
import torch
from torch.func import grad, jacrev, vmap


def f_scalar(p):            # p: (2,) → skaler
    return (p[0] + p[1]) * p[1]


def g_vec(p):               # p: (2,) → (2,)
    return torch.stack([p[0] * p[1], p[0] + p[1] ** 2])


def batch_grad(P):
    """P: (N, 2) → (N, 2): her noktada ∇f. Döngü yok."""
    ...


def batch_jacobian(P):
    """P: (N, 2) → (N, 2, 2). Döngü yok."""
    ...


P = torch.tensor([[2.0, 3.0], [0.0, 1.0]])
print(batch_grad(P))        # [[3, 8], [1, 2]]
print(batch_jacobian(P))

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Makale ödevi: Baydin ve ark. (2018): makine öğrenmesinde otomatik türev

[Baydin, Pearlmutter, Radul, Siskind — Automatic Differentiation in Machine Learning: a Survey (JMLR 2018)](https://arxiv.org/abs/1502.05767)

Makalenin Tablo 2–3'teki ileri ve geri mod izlerini kendi kodunla üret: ikili sayılar (dual numbers) ile ileri mod ve bir kayıt bandı (tape) ile geri mod yaz; $f(x_1,x_2)=\ln x_1+x_1x_2-\sin x_2$ örneğinde ara değerleri makaleyle karşılaştır. n girdi, m çıktı için Jacobian'ı ileri modda n, geri modda m geçişle hesaplayıp maliyet ödünleşimini (Bölüm 3.1–3.2) n ≫ m ve n ≪ m durumlarında ölç.


**Adımlar**

1. Dual sayı sınıfında +, ×, sin, exp, log için operatör aşırı yüklemesi.

**Kabul ölçütleri**

- İzler makaledeki tablolarla birebir.
- Jacobian maliyet tablosu: ileri mod ∝ n, geri mod ∝ m.

## Kaynaklar

- [CS231n — Backpropagation, Intuitions](https://cs231n.github.io/optimization-2/) — MIT
- [Karpathy — micrograd / Zero to Hero, 1. bölüm](https://github.com/karpathy/micrograd) — MIT
- [Mathematics for Machine Learning, Bölüm 5 (Vector Calculus)](https://mml-book.github.io/) — © (PDF ücretsiz)
- [PyTorch — torch.func (grad, jacrev, vmap)](https://docs.pytorch.org/docs/stable/func.html) — BSD-3